# Chapitre 3 - Exercices 3.21 à 3.25

| Exercice | Thème |
|---|---|
| 3.21 | Régression de rang réduit et corrélations canoniques |
| 3.22 | Invariance de la régression de rang réduit au choix de $\Sigma$ |
| 3.23 | Corrélations liées et décroissantes le long du chemin LAR |
| 3.24 | La direction LAR fait des angles égaux avec les variables actives |
| 3.25 | « Look-ahead » du LAR : quelle variable entre, et quand |

Les fonctions utilisées (CCA, régression de rang réduit, LAR…) sont dans
`esl/ch03_regression_complements.py`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import ch03_regression_complements as rc

set_style()
np.set_printoptions(precision=4, suppress=True)

## Exercice 3.21 : Régression de rang réduit

**Contexte (section 3.7).** On observe $K$ réponses : $\mathbf Y$ est $N\times K$, $\mathbf X$
est $N\times p$ (de rang $p$), les deux centrées. La **régression de rang réduit** cherche
une matrice de coefficients $\mathbf B$ ($p\times K$) de rang $m$ imposé :

$$
\hat{\mathbf B}^{rr}(m) = \operatorname*{argmin}_{\operatorname{rang}(\mathbf B)=m}\ \sum_{i=1}^N(y_i-\mathbf B^Tx_i)^T\Sigma^{-1}(y_i-\mathbf B^Tx_i)
\qquad\text{(éq. (3.68))}
$$

où $y_i, x_i$ sont les lignes de $\mathbf Y,\mathbf X$ (écrites en colonnes) et $\Sigma$ la
covariance des erreurs.

**Énoncé (reformulé).** Montrer que si $\Sigma$ est estimée par $\mathbf Y^T\mathbf Y/N$, la
solution est $\hat{\mathbf B}^{rr}(m) = \hat{\mathbf B}\,\mathbf U_m\mathbf U_m^-$ (éq. (3.69)),
où $\hat{\mathbf B}=(\mathbf X^T\mathbf X)^{-1}\mathbf X^T\mathbf Y$ est l'estimateur des moindres
carrés, $\mathbf U_m$ la matrice $K\times m$ des $m$ premiers vecteurs canoniques côté
réponses, et $\mathbf U_m^-$ un inverse généralisé. *Indication :* poser
$\mathbf Y^*=\mathbf Y\Sigma^{-1/2}$, travailler avec les vecteurs canoniques $\mathbf u_m^*$,
et montrer que $\mathbf U_m=\Sigma^{-1/2}\mathbf U_m^*$ et
$\mathbf U_m^-=\mathbf U_m^{*T}\Sigma^{1/2}$.

### Rappels : trace et norme de Frobenius

Pour une matrice $\mathbf A$ ($N\times K$), la **norme de Frobenius** est
$\|\mathbf A\|_F^2=\sum_{i,k}a_{ik}^2=\operatorname{tr}(\mathbf A^T\mathbf A)=\operatorname{tr}(\mathbf A\mathbf A^T)$.
On utilisera :

- la trace est linéaire et $\operatorname{tr}(\mathbf A\mathbf B)=\operatorname{tr}(\mathbf B\mathbf A)$ ;
- $\|\mathbf A+\mathbf C\|_F^2=\|\mathbf A\|_F^2+2\operatorname{tr}(\mathbf A^T\mathbf C)+\|\mathbf C\|_F^2$
  (Pythagore si $\operatorname{tr}(\mathbf A^T\mathbf C)=0$) ;
- pour un vecteur $z$ et une matrice symétrique $\mathbf S$, $z^T\mathbf Sz=\operatorname{tr}(\mathbf S zz^T)$.

### Rappel : meilleure approximation de rang $m$ (Eckart–Young)

**Théorème.** Soit $\mathbf M$ ($N\times K$) et $\mathbf S=\mathbf M^T\mathbf M=\sum_j\mu_j\mathbf e_j\mathbf e_j^T$
sa décomposition spectrale ($\mu_1\ge\dots\ge\mu_K\ge0$, $\mathbf e_j$ orthonormés : ce sont les
vecteurs singuliers à droite de $\mathbf M$). Parmi les matrices $\mathbf C$ de rang $\le m$,
$\|\mathbf M-\mathbf C\|_F$ est minimale pour $\mathbf C=\mathbf M\mathbf E_m\mathbf E_m^T$, où
$\mathbf E_m=[\mathbf e_1,\dots,\mathbf e_m]$.

*Preuve.* Soit $\mathbf C$ de rang $\le m$ et $\mathbf P$ le projecteur orthogonal ($K\times K$)
sur l'espace engendré par ses lignes, de dimension $\le m$.
1. Chaque ligne de $\mathbf C$ est dans $\operatorname{Im}\mathbf P$ ; la plus proche de la ligne
   correspondante de $\mathbf M$ est sa projection. Donc $\|\mathbf M-\mathbf C\|_F^2\ge\|\mathbf M-\mathbf M\mathbf P\|_F^2=\|\mathbf M\|_F^2-\operatorname{tr}(\mathbf P\mathbf S)$ (Pythagore).
2. $\operatorname{tr}(\mathbf P\mathbf S)=\sum_j\mu_j\,\mathbf e_j^T\mathbf P\mathbf e_j=\sum_j\mu_jw_j$ avec
   $w_j=\|\mathbf P\mathbf e_j\|^2\in[0,1]$ et $\sum_jw_j=\operatorname{tr}\mathbf P\le m$.
   Une somme $\sum_j\mu_jw_j$ avec de tels poids est au plus $\mu_1+\dots+\mu_m$ (on met tout
   le poids disponible sur les plus grands $\mu_j$), valeur atteinte pour $\mathbf P=\mathbf E_m\mathbf E_m^T$.

D'où $\|\mathbf M-\mathbf C\|_F^2\ge\sum_{j>m}\mu_j=\|\mathbf M-\mathbf M\mathbf E_m\mathbf E_m^T\|_F^2$. $\square$

### Étape 1 : écriture matricielle et blanchiment

Comme $z^T\Sigma^{-1}z=\|\Sigma^{-1/2}z\|^2$, le critère (3.68) s'écrit

$$
J(\mathbf B)=\sum_i\big\|\Sigma^{-1/2}(y_i-\mathbf B^Tx_i)\big\|^2=\big\|(\mathbf Y-\mathbf X\mathbf B)\Sigma^{-1/2}\big\|_F^2 ,
$$

puisque la $i$-ième ligne de $(\mathbf Y-\mathbf X\mathbf B)\Sigma^{-1/2}$ est
$(y_i-\mathbf B^Tx_i)^T\Sigma^{-1/2}$. Avec $\mathbf Y^*=\mathbf Y\Sigma^{-1/2}$ (réponses
« blanchies ») et $\mathbf B^*=\mathbf B\Sigma^{-1/2}$ :
$J(\mathbf B)=\|\mathbf Y^*-\mathbf X\mathbf B^*\|_F^2$. Comme $\Sigma^{-1/2}$ est inversible,
$\operatorname{rang}(\mathbf B^*)=\operatorname{rang}(\mathbf B)$.